In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 113.8 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Load model directly

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bhadresh-savani/distilbert-base-uncased-emotion")
model = AutoModelForSequenceClassification.from_pretrained(
    "bhadresh-savani/distilbert-base-uncased-emotion",
    num_labels=6,
    ignore_mismatched_sizes=True,
    )

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [3]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "unsplit")

README.md: 0.00B [00:00, ?B/s]

unsplit/train-00000-of-00001.parquet:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/416809 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 416809
    })
})

In [5]:
dsdf = ds['train'].to_pandas()

In [6]:
dsdf

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2
...,...,...
416804,that was what i felt when i was finally accept...,1
416805,i take every day as it comes i m just focussin...,4
416806,i just suddenly feel that everything was fake,0
416807,im feeling more eager than ever to claw back w...,1


In [7]:
dsdf.duplicated().sum()

np.int64(686)

In [8]:
dsdf.drop_duplicates(inplace=True)

In [9]:
dsdf

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2
...,...,...
416804,that was what i felt when i was finally accept...,1
416805,i take every day as it comes i m just focussin...,4
416806,i just suddenly feel that everything was fake,0
416807,im feeling more eager than ever to claw back w...,1


In [10]:
def tokenize_function(examples):
    # This automatically formats the data into input_ids and attention_mask
    return tokenizer(examples["text"], truncation=True)

In [11]:
LabelMap = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}
ReversedLabelMap ={
    'sadness': 0,
    'joy': 0,
    'love': 0,
    'anger': 0,
    'fear': 0,
    'surprise': 0
}

In [12]:
from datasets import Dataset
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Convert pandas DataFrame back to Hugging Face Dataset
final_ds = Dataset.from_pandas(dsdf, preserve_index=False)

# Apply formatting across the dataset using batched=True
tokenized_datasets = final_ds.map(tokenize_function, batched=True, cache_file_name="/kaggle/working/tokenized_cache.arrow")
# Set the format to PyTorch tensors
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

display(tokenized_datasets)

Map:   0%|          | 0/416123 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 416123
})

### Prepare Training and Validation Splits
We'll split the tokenized dataset into a training set and a test set to monitor performance during training.

In [13]:
# Split the dataset: 80% train, 20% test
train_test_split = tokenized_datasets.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Training samples: 374510
Validation samples: 41613


### Define Training Arguments and Initialize Trainer
We will configure the `TrainingArguments` and use the `Trainer` class. Note: This model is quite large, so we use a small batch size to fit in Colab's GPU memory.

In [14]:
!pip install evaluate
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="mental-roberta-emotion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    push_to_hub=False,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Start training
trainer.train()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.198476,0.184859,0.941965
2,0.166637,0.175380,0.942686
3,0.145010,0.180991,0.942734


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17556, training_loss=0.18160087412053888, metrics={'train_runtime': 4146.301, 'train_samples_per_second': 270.972, 'train_steps_per_second': 4.234, 'total_flos': 1.6584860967637464e+16, 'train_loss': 0.18160087412053888, 'epoch': 3.0})

In [15]:
 # Evaluate the model on the validation split
# eval_results = trainer.evaluate()
# print(f"Final Validation Results: {eval_results}")

# Save the model and tokenizer to a local folder
model.save_pretrained("./final_mental_emotion_model")
tokenizer.save_pretrained("./final_mental_emotion_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_mental_emotion_model/tokenizer_config.json',
 './final_mental_emotion_model/tokenizer.json')

In [16]:
from transformers import pipeline

# Create a classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

# Testing examples for each of the 6 classes
test_sentences = [
    "i feel awful about it too because it s my job to help her.",       # sadness
    "i feel like i have performed well and achieved a huge milestone.",  # joy
    "i feel romantic and nostalgic when thinking about our time.",       # love
    "i feel so frustrated and irritated by the way things are handled.", # anger
    "i remember feeling acutely distressed and anxious.",                # fear
    "i keep feeling pleasantly surprised at the unexpected support."     # surprise
]

print("\n--- Inference Results ---")
for text in test_sentences:
    result = classifier(text)[0]
    raw_label = result['label']

    # Check if the label is a generic ID (e.g., 'LABEL_0') or the actual name
    if raw_label.startswith("LABEL_"):
        label_id = int(raw_label.split('_')[-1])
        emotion_name = LabelMap.get(label_id, "Unknown")
    else:
        # If it's already 'sadness', 'joy', etc., use it directly
        emotion_name = raw_label

    print(f"Text: {text}")
    print(f"Predicted: {emotion_name.upper()} (Confidence: {result['score']:.4f})\n")


--- Inference Results ---
Text: i feel awful about it too because it s my job to help her.
Predicted: SADNESS (Confidence: 1.0000)

Text: i feel like i have performed well and achieved a huge milestone.
Predicted: JOY (Confidence: 1.0000)

Text: i feel romantic and nostalgic when thinking about our time.
Predicted: LOVE (Confidence: 0.9999)

Text: i feel so frustrated and irritated by the way things are handled.
Predicted: ANGER (Confidence: 1.0000)

Text: i remember feeling acutely distressed and anxious.
Predicted: FEAR (Confidence: 1.0000)

Text: i keep feeling pleasantly surprised at the unexpected support.
Predicted: SURPRISE (Confidence: 0.9999)

